In [1]:
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.response.FullDetectorResponse import FullDetectorResponse
from cosipy.util import fetch_wasabi_file
from histpy import Histogram

from scoords import SpacecraftFrame

from astropy.time import Time
import astropy.units as u

SED_KEV_TO_ERG = u.keV.to(u.erg)
KEV_TO_MEV = u.keV.to(u.MeV)
from astropy.coordinates import SkyCoord, Galactic

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from threeML import *
from threeML.io.package_data import get_path_of_data_file
from threeML.io.logging import silence_console_log
from astromodels import Parameter
from threeML.minimizer.minimization import CannotComputeCovariance

from jupyterthemes import jtplot
jtplot.style(context="talk", fscale=1, ticks=True, grid=False)
set_threeML_style()
silence_warnings()

from scipy.integrate import quad

import matplotlib.ticker as mticker

from pathlib import Path

import os

# Use finite background support consistently in every likelihood.
BACKGROUND_PSEUDOCOUNT = 1e-12

%matplotlib inline

13:38:39 WARNING   The naima package is not available. Models that depend on it will not be         ]8;id=968038;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=888548;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#43\43]8;;\
                  available                                                                                        

         WARNING   The GSL library or the pygsl wrapper cannot be loaded. Models that depend on it  ]8;id=56358;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=833100;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#65\65]8;;\
                  will not be available.                                                                           

         WARNING   The ebltable package is not available. Models that depend on it will not be     ]8;id=190235;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/absorption.py\absorption.py]8;;\:]8;id=668078;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/absorption.py#33\33]8;;\
                  available                                                                                        

13:38:40 INFO      Starting 3ML!                                                                     ]8;id=9361;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=784512;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#44\44]8;;\

         WARNING   WARNINGs here are NOT errors                                                      ]8;id=544799;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=550725;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#45\45]8;;\

...

         WARNING   Could not import plugin FermiLATLike.py. Do you have the relative instrument     ]8;id=244549;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=873468;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#126\126]8;;\
                  software installed and configured?                                                               

         WARNING   No fermitools installed                                              ]8;id=484946;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py\lat_transient_builder.py]8;;\:]8;id=601514;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py#44\44]8;;\

         WARNING   Env. variable OMP_NUM_THREADS is not set. Please set it to 1 for optimal         ]8;id=713770;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=155061;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

         WARNING   Env. variable MKL_NUM_THREADS is not set. Please set it to 1 for optimal         ]8;id=40176;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=540878;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

         WARNING   Env. variable NUMEXPR_NUM_THREADS is not set. Please set it to 1 for optimal     ]8;id=837783;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=55133;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

In [40]:
# Simulation paths are defined with the histogram inputs below.

In [2]:
from astropy import units as u
from astropy.coordinates import SkyCoord
from cosipy.event_selection import GoodTimeInterval
from agn_cosi_fit_utils import open_spacecraft_history, scale_spacecraft_livetime

orientation_path = "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/DC4_Files/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits"
source_coord = SkyCoord(l=155.077, b=75.063, frame="galactic", unit="deg")
fov_cut = 60 * u.deg

full_sc_orientation = open_spacecraft_history(orientation_path)
source_gti = GoodTimeInterval.from_pointing_cut(
    source_coord,
    full_sc_orientation,
    fov_cut,
    earth_occ=False,
)
sc_orientation = full_sc_orientation.apply_gti(source_gti)

print(f"NGC 4151 FOV cut: {fov_cut.to_value(u.deg):.0f} deg")
print(f"Selected livetime: {sc_orientation.cumulative_livetime().to_value(u.s):,.1f} s")

NGC 4151 FOV cut: 60 deg
Selected livetime: 980,415.0 s


In [3]:
dr = "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/DC4_Files/ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5"

In [4]:
multiplier_4151 = 8

exposure_4151 = multiplier_4151 * 3

# Scale count histograms and response livetime together; keep source flux intrinsic.
sc_orientation = scale_spacecraft_livetime(sc_orientation, multiplier_4151)

# Source Model Plots

## Cutoff Power Law (Thermal)

#### NGC 4151 (200 keV)

In [5]:
K_inj = 0.12 / u.cm / u.cm / u.s / u.keV
piv_inj = 1. * u.keV
xc_inj = 203. * u.keV
index_inj = -1.74

spectrum_inj_ec200 = Cutoff_powerlaw()

spectrum_inj_ec200.K.value = K_inj.value
spectrum_inj_ec200.piv.value = piv_inj.value
spectrum_inj_ec200.xc.value = xc_inj.value
spectrum_inj_ec200.index.value = index_inj

spectrum_inj_ec200.K.unit = K_inj.unit
spectrum_inj_ec200.piv.unit = piv_inj.unit
spectrum_inj_ec200.xc.unit = xc_inj.unit

#### NGC 4151 (1000 keV)

In [6]:
K_inj = 0.12 / u.cm / u.cm / u.s / u.keV
piv_inj = 1. * u.keV
xc_inj = 1000. * u.keV
index_inj = -1.74

spectrum_inj_ec1000 = Cutoff_powerlaw()

spectrum_inj_ec1000.K.value = K_inj.value
spectrum_inj_ec1000.piv.value = piv_inj.value
spectrum_inj_ec1000.xc.value = xc_inj.value
spectrum_inj_ec1000.index.value = index_inj

spectrum_inj_ec1000.K.unit = K_inj.unit
spectrum_inj_ec1000.piv.unit = piv_inj.unit
spectrum_inj_ec1000.xc.unit = xc_inj.unit

# Power law tail (Non-thermal)

#### NGC 4151 (200 keV)

In [7]:
# K_inj = 0.1*spectrum_inj_ec200.evaluate_at(1000) / u.cm / u.cm / u.s / u.keV
# piv_inj = 1000. * u.keV

def cutoff_powerlaw_k_at_pivot(shape, pivot_value):
    return float(shape.K.value * (pivot_value / shape.piv.value) ** shape.index.value)

K_inj = 0.65*spectrum_inj_ec200.evaluate_at(200) / u.cm / u.cm / u.s / u.keV
piv_inj = 200. * u.keV
index_inj = -2.8

spectrum_inj_ec200_PL = Powerlaw()

spectrum_inj_ec200_PL.K.value = K_inj.value
spectrum_inj_ec200_PL.piv.value = piv_inj.value
spectrum_inj_ec200_PL.index.value = index_inj

spectrum_inj_ec200_PL.K.unit = K_inj.unit
spectrum_inj_ec200_PL.piv.unit = piv_inj.unit

spectrum_inj_ec200_total = spectrum_inj_ec200 + spectrum_inj_ec200_PL

print("K_inj: ", K_inj)
print(spectrum_inj_ec200.evaluate_at(200))
print(spectrum_inj_ec200_PL.evaluate_at(200))
print("="*40)
linking_ratio_ec200 = spectrum_inj_ec200_PL.K.value / cutoff_powerlaw_k_at_pivot(
    spectrum_inj_ec200,
    spectrum_inj_ec200_PL.piv.value,
)
print("Linking K ratio", linking_ratio_ec200)
print("Flux ratio at 200 keV", spectrum_inj_ec200_PL.evaluate_at(200)/spectrum_inj_ec200.evaluate_at(200))
print("="*40)
flux_th, _ = quad(spectrum_inj_ec200.evaluate_at, 200.0, 5000.0)
flux_nth, _ = quad(spectrum_inj_ec200_PL.evaluate_at, 200.0, 5000.0)
print("Non-thermal flux: ", flux_nth)
print("Total flux: ", flux_th + flux_nth)
print("Thermal - Non-thermal Flux Ratio: ", flux_th/flux_nth)
print("Non-thermal percentage: ", flux_nth/(flux_th + flux_nth))

K_inj:  3.3721558756085362e-06 1 / (keV s cm2)
5.187932116320825e-06
3.3721558756085362e-06
Linking K ratio 0.23912163676143755
Flux ratio at 200 keV 0.65
Non-thermal flux:  0.00037354275634502636
Total flux:  0.0008309220159224175
Thermal - Non-thermal Flux Ratio:  1.2244361637545138
Non-thermal percentage:  0.4495521230477347


#### NGC 4151 (1000 keV)

In [8]:
# K_inj = (spectrum_inj_ec1000.evaluate_at(3000)) / u.cm / u.cm / u.s / u.keV
K_inj = 0.12*spectrum_inj_ec1000.evaluate_at(200) / u.cm / u.cm / u.s / u.keV
piv_inj = 200. * u.keV
index_inj = -2.8

spectrum_inj_ec1000_PL = Powerlaw()

spectrum_inj_ec1000_PL.K.value = K_inj.value
spectrum_inj_ec1000_PL.piv.value = piv_inj.value
spectrum_inj_ec1000_PL.index.value = index_inj

spectrum_inj_ec1000_PL.K.unit = K_inj.unit
spectrum_inj_ec1000_PL.piv.unit = piv_inj.unit

spectrum_inj_ec1000_total = spectrum_inj_ec1000 + spectrum_inj_ec1000_PL

print("K_inj: ", K_inj)
print(spectrum_inj_ec1000.evaluate_at(200))
print(spectrum_inj_ec1000_PL.evaluate_at(200))
print("="*40)
linking_ratio_ec1000 = spectrum_inj_ec1000_PL.K.value / cutoff_powerlaw_k_at_pivot(
    spectrum_inj_ec1000,
    spectrum_inj_ec1000_PL.piv.value,
)
print("Linking K ratio", linking_ratio_ec1000)
print("Flux ratio at 200 keV", spectrum_inj_ec1000_PL.evaluate_at(200)/spectrum_inj_ec1000.evaluate_at(200))
print("="*40)
flux_th, _ = quad(spectrum_inj_ec1000.evaluate_at, 200.0, 5000.0)
flux_nth, _ = quad(spectrum_inj_ec1000_PL.evaluate_at, 200.0, 5000.0)
print("Non-thermal flux: ", flux_nth)
print("Total flux: ", flux_th + flux_nth)
print("Thermal - Non-thermal Flux Ratio: ", flux_th/flux_nth)
print("Non-thermal percentage: ", flux_nth/(flux_th + flux_nth))

K_inj:  1.3855146310935058e-06 1 / (keV s cm2)
1.1545955259112548e-05
1.3855146310935069e-06
Linking K ratio 0.09824769036935797
Flux ratio at 200 keV 0.12000000000000009
Non-thermal flux:  0.00015347717405312245
Total flux:  0.002049009517086617
Thermal - Non-thermal Flux Ratio:  12.350581477199999
Non-thermal percentage:  0.07490310453576804


# Spectral Fitting

In [9]:
# Load the representative median realization selected by the sensitivity ensemble.
import json

sensitivity_dir = Path("/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/GammaRay/Paper_Models/Fluctuate_True/Sensitivity_Ensemble")
ec200_manifest_file = sensitivity_dir / "Case2_ec200" / f"NGC4151_ec203_K012_idx174_NTH_threshold_median_realization_{exposure_4151}months.json"
if not ec200_manifest_file.exists():
    raise FileNotFoundError(f"Run the Case2_ec200 sensitivity notebook first: {ec200_manifest_file}")
with ec200_manifest_file.open() as stream:
    ec200_manifest = json.load(stream)
NGC4151_ec200 = Histogram.open(ec200_manifest["source_expectation_file"])
NGC4151_ec200_bkg = Histogram.open(ec200_manifest["median_data_file"])

bkg = Histogram.open(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/"
    "COSI/Radio_Quiet_AGN/DC4_Files/Background/"
    "Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck_"
    "NGC4151_60deg_fov_cut.hdf5"
) * multiplier_4151

# Collapse the background time axis.
bkg = bkg.project("Em", "Phi", "PsiChi")

# Match metadata, following the source-injector branch workflow.
NGC4151_ec200.axes["Em"].axis_scale = bkg.axes["Em"].axis_scale
NGC4151_ec200 = NGC4151_ec200.to(
    unit=bkg.unit,
    update=False,
)

NGC4151_ec200_bkg.axes["Em"].axis_scale = bkg.axes["Em"].axis_scale
NGC4151_ec200_bkg = NGC4151_ec200_bkg.to(unit=bkg.unit, update=False)

In [10]:
ec1000_manifest_file = sensitivity_dir / "Case2_ec1000" / f"NGC4151_ec1000_K012_idx174_NTH_threshold_median_realization_{exposure_4151}months.json"
if not ec1000_manifest_file.exists():
    raise FileNotFoundError(f"Run the Case2_ec1000 sensitivity notebook first: {ec1000_manifest_file}")
with ec1000_manifest_file.open() as stream:
    ec1000_manifest = json.load(stream)
NGC4151_ec1000 = Histogram.open(ec1000_manifest["source_expectation_file"])
NGC4151_ec1000_bkg = Histogram.open(ec1000_manifest["median_data_file"])

# Match metadata, following the source-injector branch workflow.
NGC4151_ec1000.axes["Em"].axis_scale = bkg.axes["Em"].axis_scale
NGC4151_ec1000 = NGC4151_ec1000.to(
    unit=bkg.unit,
    update=False,
)

NGC4151_ec1000_bkg.axes["Em"].axis_scale = bkg.axes["Em"].axis_scale
NGC4151_ec1000_bkg = NGC4151_ec1000_bkg.to(unit=bkg.unit, update=False)

## Perform spectral fit

Set background parameter, which is used to fit the amplitude of the background, and instantiate the COSI 3ML plugin

In [11]:
from agn_cosi_fit_utils import COSIPlugin, EnergyRangeCOSIPlugin, make_cosi_background_parameter
bkg_par = Parameter("background_cosi",                                         # background parameter
                     1,                                                        # initial value of parameter
                     min_value=0,                                              # minimum value of parameter
                     max_value=5,                                              # maximum value of parameter
                     delta=0.05,                                               # initial step used by fitting engine
                     desc="Background parameter for cosi")

bkg_par_ec1000 = Parameter("background_cosi_ec1000",                           # background parameter
                     1,                                                        # initial value of parameter
                     min_value=0,                                              # minimum value of parameter
                     max_value=5,                                              # maximum value of parameter
                     delta=0.05,                                               # initial step used by fitting engine
                     desc="Background parameter for cosi")

cosi = COSIPlugin("cosi",                                                        # COSI 3ML plugin
                 dr = dr,                                                      # detector response
                 data = NGC4151_ec200_bkg.project('Em', 'Phi', 'PsiChi'),   # data (source+background)
                 bkg = bkg.project('Em', 'Phi', 'PsiChi'),         # background model 
                 sc_orientation = sc_orientation,                              # spacecraft orientation
                 nuisance_param = bkg_par,                                     # background parameter
                 background_pseudocount = BACKGROUND_PSEUDOCOUNT,              # finite support for empty background cells
                 earth_occ = True)                                             # Option to account for Earth occultation

cosi_ec1000 = COSIPlugin("cosi",                                                        # COSI 3ML plugin
                 dr = dr,                                                      # detector response
                 data = NGC4151_ec1000_bkg.project('Em', 'Phi', 'PsiChi'),   # data (source+background)
                 bkg = bkg.project('Em', 'Phi', 'PsiChi'),         # background model 
                 sc_orientation = sc_orientation,                              # spacecraft orientation
                 nuisance_param = bkg_par_ec1000,                                     # background parameter
                 background_pseudocount = BACKGROUND_PSEUDOCOUNT,              # finite support for empty background cells
                 earth_occ = True)                                             # Option to account for Earth occultation

### Powerlaw with energy cutoff fit

In [12]:
l=155.07
b=75.06

# Give it some harsher initial guesses
K = (spectrum_inj_ec200.K.value * (200.0 / spectrum_inj_ec200.piv.value) ** spectrum_inj_ec200.index.value) / u.cm / u.cm / u.s / u.keV
piv = 200. * u.keV
xc = 100. * u.keV
index = -1.74

spectrum_cpl = Cutoff_powerlaw()

spectrum_cpl.K.value = K.value
spectrum_cpl.piv.value = piv.value
spectrum_cpl.xc.value = xc.value
spectrum_cpl.index.value = index
spectrum_cpl.index.fix = True

# Harsher Parameters
spectrum_cpl.K.min_value = 1e-8
spectrum_cpl.K.max_value = 1e-2
spectrum_cpl.xc.min_value = 100 # keep these relatively the same
spectrum_cpl.xc.max_value = 10000 # change to 1000
# spectrum_cpl.index.min_value = -3 # change to -3.5 to sample lower, larger values (-5, 5)
# spectrum_cpl.index.max_value = 1

# spectrum_cpl.K.delta = 0.05
# spectrum_cpl.xc.delta = 10
# spectrum_cpl.index.delta = 0.15

spectrum_cpl.K.unit = K.unit
spectrum_cpl.piv.unit = piv.unit
spectrum_cpl.xc.unit = xc.unit



In [13]:
l=155.07
b=75.06

# Give it some harsher initial guesses
K = (spectrum_inj_ec1000.K.value * (200.0 / spectrum_inj_ec1000.piv.value) ** spectrum_inj_ec1000.index.value) / u.cm / u.cm / u.s / u.keV
piv = 200. * u.keV
xc = 1000. * u.keV
index = -1.74

spectrum_cpl_ec1000 = Cutoff_powerlaw()

spectrum_cpl_ec1000.K.value = K.value
spectrum_cpl_ec1000.piv.value = piv.value
spectrum_cpl_ec1000.xc.value = xc.value
spectrum_cpl_ec1000.index.value = index
spectrum_cpl_ec1000.index.fix = True

# Harsher Parameters
spectrum_cpl_ec1000.K.min_value = 1e-8
spectrum_cpl_ec1000.K.max_value = 1e-2
spectrum_cpl_ec1000.xc.min_value = 100 # keep these relatively the same
spectrum_cpl_ec1000.xc.max_value = 10000 # change to 1000
# spectrum_cpl_ec1000.index.min_value = -3 # change to -3.5 to sample lower, larger values (-5, 5)
# spectrum_cpl_ec1000.index.max_value = 1

# spectrum_cpl_ec1000.K.delta = 0.05
# spectrum_cpl_ec1000.xc.delta = 10
# spectrum_cpl_ec1000.index.delta = 0.15

spectrum_cpl_ec1000.K.unit = K.unit
spectrum_cpl_ec1000.piv.unit = piv.unit
spectrum_cpl_ec1000.xc.unit = xc.unit



## Thermal + Non-thermal Fit

In [14]:
l=155.07
b=75.06

# Give it some harsher initial guesses
K = 1e-5 / u.cm / u.cm / u.s / u.keV
piv = 200. * u.keV
index = -2.8

spectrum = Powerlaw()

spectrum.K.value = K.value
spectrum.piv.value = piv.value
spectrum.index.value = index
# spectrum.index.fix = True

# Harsher Parameters
spectrum.K.min_value = 1e-8
spectrum.K.max_value = 1e-2
spectrum.index.min_value = -5
spectrum.index.max_value = 1

# spectrum.K.delta = 5
spectrum.index.delta = 0.25

spectrum.K.unit = K.unit
spectrum.piv.unit = piv.unit



In [15]:
l=155.07
b=75.06

# Give it some harsher initial guesses
K = 1e-5 / u.cm / u.cm / u.s / u.keV
piv = 200. * u.keV
index = -2.8

spectrum_ec1000 = Powerlaw()

spectrum_ec1000.K.value = K.value
spectrum_ec1000.piv.value = piv.value
spectrum_ec1000.index.value = index
# spectrum_ec1000.index.fix = True

# Harsher Parameters
spectrum_ec1000.K.min_value = 1e-8
spectrum_ec1000.K.max_value = 1e-2
spectrum_ec1000.index.min_value = -5
spectrum_ec1000.index.max_value = 1

# spectrum_ec1000.K.delta = 5
spectrum_ec1000.index.delta = 0.25

spectrum_ec1000.K.unit = K.unit
spectrum_ec1000.piv.unit = piv.unit



In [16]:
from agn_cosi_fit_utils import COSIPlugin, EnergyRangeCOSIPlugin, make_cosi_background_parameter
# Keep both spectral components in one point source so the response cache
# tracks every component as linked or independently fitted parameters change.
source1 = PointSource(
    "source1",
    l=l,
    b=b,
    spectral_shape=spectrum_cpl + spectrum,
)
source2 = PointSource(
    "source2",
    l=l,
    b=b,
    spectral_shape=spectrum_cpl_ec1000 + spectrum_ec1000,
)

model = Model(source1)
model_ec1000 = Model(source2)

cosi.set_model(model)
cosi_ec1000.set_model(model_ec1000)


def make_cpl_only_source(name, reference_source):
    reference_shape = reference_source.spectrum.main.shape.functions[0]
    cpl_shape = Cutoff_powerlaw()

    for parameter_name in ("K", "piv", "xc", "index"):
        reference_parameter = getattr(reference_shape, parameter_name)
        cpl_parameter = getattr(cpl_shape, parameter_name)
        cpl_parameter.value = reference_parameter.value
        cpl_parameter.fix = reference_parameter.fix

        if reference_parameter.min_value is not None:
            cpl_parameter.min_value = reference_parameter.min_value
        if reference_parameter.max_value is not None:
            cpl_parameter.max_value = reference_parameter.max_value
        if reference_parameter.delta is not None:
            cpl_parameter.delta = reference_parameter.delta
        if reference_parameter.unit is not None:
            cpl_parameter.unit = reference_parameter.unit
    return PointSource(
        name,
        l=l,
        b=b,
        spectral_shape=cpl_shape,
    )


source1_cpl_only = make_cpl_only_source("source1_cpl_only", source1)
source2_cpl_only = make_cpl_only_source("source2_cpl_only", source2)

model_cpl_only = Model(source1_cpl_only)
model_ec1000_cpl_only = Model(source2_cpl_only)

bkg_par_cpl_only = Parameter(
    "background_cosi_cpl_only",
    1,
    min_value=0,
    max_value=5,
    delta=0.05,
    desc="Background parameter for the Cpl-only COSI fit",
)

bkg_par_ec1000_cpl_only = Parameter(
    "background_cosi_ec1000_cpl_only",
    1,
    min_value=0,
    max_value=5,
    delta=0.05,
    desc="Background parameter for the Cpl-only COSI fit",
)

cosi_cpl_only = COSIPlugin(
    "cosi_cpl_only",
    dr=dr,
    data=NGC4151_ec200_bkg.project("Em", "Phi", "PsiChi"),
    bkg=bkg.project("Em", "Phi", "PsiChi"),
    sc_orientation=sc_orientation,
    nuisance_param=bkg_par_cpl_only,
    background_pseudocount=BACKGROUND_PSEUDOCOUNT,
    earth_occ=True,
)

cosi_ec1000_cpl_only = COSIPlugin(
    "cosi_ec1000_cpl_only",
    dr=dr,
    data=NGC4151_ec1000_bkg.project("Em", "Phi", "PsiChi"),
    bkg=bkg.project("Em", "Phi", "PsiChi"),
    sc_orientation=sc_orientation,
    nuisance_param=bkg_par_ec1000_cpl_only,
    background_pseudocount=BACKGROUND_PSEUDOCOUNT,
    earth_occ=True,
)

cosi_cpl_only.set_model(model_cpl_only)
cosi_ec1000_cpl_only.set_model(model_ec1000_cpl_only)


### Joint Fit

In [ ]:
# bat_ec200.use_effective_area_correction(0.02, 1.8)
# plugins = DataList(bat_ec200, cosi)

# bat_ec1000.use_effective_area_correction(0.02, 1.8)
# plugins_ec1000 = DataList(bat_ec1000, cosi_ec1000)

### Only COSI Fit

In [17]:
## Only COSI fit
plugins = DataList(cosi)
plugins_ec1000 = DataList(cosi_ec1000)
plugins_cpl_only = DataList(cosi_cpl_only)
plugins_ec1000_cpl_only = DataList(cosi_ec1000_cpl_only)

### Cpl + Pl vs Cpl-only comparison

In [ ]:
ratio_ec200 = linking_ratio_ec200
link_function = Line(a=0.0, b=ratio_ec200)   # tail K = a + b * source1.K
link_function.a.fix = True
link_function.b.min_value = 0.0  # Require a non-negative non-thermal normalization.
# link_function.b.fix = True             # keep ratio fixed at

ratio_ec1000 = linking_ratio_ec1000
link_function_ec1000 = Line(a=0.0, b=ratio_ec1000)   # tail K = a + b * source2.K
link_function_ec1000.a.fix = True
link_function_ec1000.b.min_value = 0.0  # Require a non-negative non-thermal normalization.
# link_function_ec1000.b.fix = True             # keep ratio fixed

model.link(
    model.source1.spectrum.main.composite.K_2,         # dependent
    model.source1.spectrum.main.composite.K_1,  # independent
    link_function,
)
model_ec1000.link(
    model_ec1000.source2.spectrum.main.composite.K_2,         # dependent
    model_ec1000.source2.spectrum.main.composite.K_1,  # independent
    link_function_ec1000,
)

like = JointLikelihood(model, plugins, verbose=False)
like_ec1000 = JointLikelihood(model_ec1000, plugins_ec1000, verbose=False)

like_cpl_only = JointLikelihood(model_cpl_only, plugins_cpl_only, verbose=False)
like_ec1000_cpl_only = JointLikelihood(
    model_ec1000_cpl_only,
    plugins_ec1000_cpl_only,
    verbose=False,
)

result = like.fit()
result_ec1000 = like_ec1000.fit()

result_cpl_only = like_cpl_only.fit()
result_ec1000_cpl_only = like_ec1000_cpl_only.fit()


In [19]:
from agn_cosi_fit_utils import COSIPlugin, EnergyRangeCOSIPlugin, make_cosi_background_parameter
def make_null_likelihood(data_hist, label):
    bkg_par_null = Parameter(
        f"background_cosi_null_{label}",
        1,
        min_value=0,
        max_value=5,
        delta=0.05,
        desc="Background parameter for the null COSI fit",
    )

    cosi_null = COSIPlugin(
        f"cosi_null_{label}",
        dr=dr,
        data=data_hist.project("Em", "Phi", "PsiChi"),
        bkg=bkg.project("Em", "Phi", "PsiChi"),
        sc_orientation=sc_orientation,
        nuisance_param=bkg_par_null,
        background_pseudocount=BACKGROUND_PSEUDOCOUNT,
        earth_occ=True,
    )

    spectrum_null = Powerlaw()
    spectrum_null.K.value = 1e-30
    spectrum_null.index.value = 1
    spectrum_null.K.fix = True
    spectrum_null.index.fix = True

    source_null = PointSource(
        "source_null",
        l=l,
        b=b,
        spectral_shape=spectrum_null,
    )

    model_null = Model(source_null)
    cosi_null.set_model(model_null)

    plugins_null = DataList(cosi_null)
    like_null = JointLikelihood(model_null, plugins_null, verbose=False)
    like_null.fit()

    return like_null


def get_likelihood_statistic(joint_likelihood):
    statistic_frame = joint_likelihood.results.get_statistic_frame()
    statistic_series = statistic_frame["-log(likelihood)"]

    if "total" in statistic_series.index:
        return float(statistic_series.loc["total"])

    return float(statistic_series.sum())


def detection_ts(null_likelihood, source_likelihood):
    return 2.0 * (
        get_likelihood_statistic(null_likelihood)
        - get_likelihood_statistic(source_likelihood)
    )


def model_improvement_ts(reference_likelihood, test_likelihood):
    return 2.0 * (
        get_likelihood_statistic(reference_likelihood)
        - get_likelihood_statistic(test_likelihood)
    )


like_null_200 = make_null_likelihood(NGC4151_ec200_bkg, "ec200")
like_null_1000 = make_null_likelihood(NGC4151_ec1000_bkg, "ec1000")

TS_cpl_200 = detection_ts(like_null_200, like_cpl_only)
TS_cpl_pl_200 = detection_ts(like_null_200, like)
TS_cpl_pl_vs_cpl_200 = model_improvement_ts(like_cpl_only, like)

TS_cpl_1000 = detection_ts(like_null_1000, like_ec1000_cpl_only)
TS_cpl_pl_1000 = detection_ts(like_null_1000, like_ec1000)
TS_cpl_pl_vs_cpl_1000 = model_improvement_ts(like_ec1000_cpl_only, like_ec1000)

fit_ts_comparison = pd.DataFrame(
    [
        {
            "spectrum": "NGC 4151 Ec=203 keV",
            "TS_Cpl": TS_cpl_200,
            "TS_Cpl_plus_Pl": TS_cpl_pl_200,
            "Delta_TS_Cpl_plus_Pl_vs_Cpl": TS_cpl_pl_vs_cpl_200,
            "Sigma_Cpl": np.sqrt(max(TS_cpl_200, 0.0)),
            "Sigma_Cpl_plus_Pl": np.sqrt(max(TS_cpl_pl_200, 0.0)),
            "Sigma_added_Pl": np.sqrt(max(TS_cpl_pl_vs_cpl_200, 0.0)),
        },
        {
            "spectrum": "NGC 4151 Ec=1000 keV",
            "TS_Cpl": TS_cpl_1000,
            "TS_Cpl_plus_Pl": TS_cpl_pl_1000,
            "Delta_TS_Cpl_plus_Pl_vs_Cpl": TS_cpl_pl_vs_cpl_1000,
            "Sigma_Cpl": np.sqrt(max(TS_cpl_1000, 0.0)),
            "Sigma_Cpl_plus_Pl": np.sqrt(max(TS_cpl_pl_1000, 0.0)),
            "Sigma_added_Pl": np.sqrt(max(TS_cpl_pl_vs_cpl_1000, 0.0)),
        },
    ]
)

display(fit_ts_comparison)

for _, row in fit_ts_comparison.iterrows():
    print(row["spectrum"])
    print(f"  Cpl TS: {row['TS_Cpl']:.3f} ({row['Sigma_Cpl']:.2f} sigma)")
    print(f"  Cpl + Pl TS: {row['TS_Cpl_plus_Pl']:.3f} ({row['Sigma_Cpl_plus_Pl']:.2f} sigma)")
    print(
        "  Added Pl improvement TS: "
        f"{row['Delta_TS_Cpl_plus_Pl_vs_Cpl']:.3f} "
        f"({row['Sigma_added_Pl']:.2f} sigma)"
    )

# Backward-compatible aliases used by the plotting cells below.
TS = TS_cpl_pl_200
TS_ec1000 = TS_cpl_pl_1000

13:39:55 INFO      set the minimizer to minuit                                             ]8;id=722960;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=97595;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

Best fit values:

,result,unit
parameter,,
background_cosi_null_ec200,(2.56750 +/- 0.00018) x 10,Hz


Correlation matrix:

1.00


...

,-log(likelihood)
cosi_null_ec1000,-1324714480.191567
total,-1324714480.191567


Values of statistical measures:

,statistical measures
AIC,-2649428958.3831167
BIC,-2649428948.0355616


,spectrum,TS_Cpl,TS_Cpl_plus_Pl,Delta_TS_Cpl_plus_Pl_vs_Cpl,Sigma_Cpl,Sigma_Cpl_plus_Pl,Sigma_added_Pl
0,NGC 4151 Ec=200 keV,7320.302002,7329.585650,9.283648,85.558763,85.612999,3.046908
1,NGC 4151 Ec=1000 keV,18001.536332,18007.355745,5.819413,134.169804,134.191489,2.412346


NGC 4151 Ec=200 keV
  Cpl TS: 7320.302 (85.56 sigma)
  Cpl + Pl TS: 7329.586 (85.61 sigma)
  Added Pl improvement TS: 9.284 (3.05 sigma)
NGC 4151 Ec=1000 keV
  Cpl TS: 18001.536 (134.17 sigma)
  Cpl + Pl TS: 18007.356 (134.19 sigma)
  Added Pl improvement TS: 5.819 (2.41 sigma)


In [ ]:
from agn_cosi_fit_utils import (
    cosi_source_detection_ts,
    fit_cosi_exposure_from_results,
    load_agn_manifest_histograms,
    save_agn_fit_summary,
)

FIT_SUMMARY_DIR = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/"
    "Papers/AGN_Corona_EC_New/Fits/Fluctuate_True"
)
DUAL_EXPOSURE_BACKGROUND_FILE = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/"
    "COSI/Radio_Quiet_AGN/DC4_Files/Background/"
    "Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck_"
    "NGC4151_60deg_fov_cut.hdf5"
)

def _dual_exposure_orientation(exposure_months):
    return scale_spacecraft_livetime(
        full_sc_orientation.apply_gti(source_gti), exposure_months / 3.0,
    )

from agn_cosi_fit_utils import model_improvement_ts as exposure_model_improvement_ts

# Refit the full and Cpl-only models to matched three-month companions.
def _case2_companion_path(case_folder, output_prefix):
    return sensitivity_dir / case_folder / (
        f"{output_prefix}_threshold_companion_realization_3months.json"
    )

_, _, ec200_data_3m, bkg_3m = load_agn_manifest_histograms(
    _case2_companion_path("Case2_ec200", "NGC4151_ec203_K012_idx174_NTH"),
    DUAL_EXPOSURE_BACKGROUND_FILE, 3,
)
_, _, ec1000_data_3m, _ = load_agn_manifest_histograms(
    _case2_companion_path("Case2_ec1000", "NGC4151_ec1000_K012_idx174_NTH"),
    DUAL_EXPOSURE_BACKGROUND_FILE, 3,
)
sc_orientation_3m = _dual_exposure_orientation(3)

def _refit_case2(reference_results, data_hist, label):
    return fit_cosi_exposure_from_results(
        reference_results=reference_results, data_hist=data_hist,
        background_hist=bkg_3m, dr_path=dr, sc_orientation=sc_orientation_3m,
        dataset_name=label,
        background_pseudocount=BACKGROUND_PSEUDOCOUNT,
    )

like_ec200_3m, cosi_ec200_3m = _refit_case2(like.results, ec200_data_3m, "cosi_case2_ec200_cpl_pl_3m")
like_ec200_cpl_3m, cosi_ec200_cpl_3m = _refit_case2(like_cpl_only.results, ec200_data_3m, "cosi_case2_ec200_cpl_3m")
like_ec1000_3m, cosi_ec1000_3m = _refit_case2(like_ec1000.results, ec1000_data_3m, "cosi_case2_ec1000_cpl_pl_3m")
like_ec1000_cpl_3m, cosi_ec1000_cpl_3m = _refit_case2(like_ec1000_cpl_only.results, ec1000_data_3m, "cosi_case2_ec1000_cpl_3m")

fit_pairs = {
    "3m_ec200_cpl_pl": (like_ec200_3m, cosi_ec200_3m, 3),
    "3m_ec200_cpl": (like_ec200_cpl_3m, cosi_ec200_cpl_3m, 3),
    "3m_ec1000_cpl_pl": (like_ec1000_3m, cosi_ec1000_3m, 3),
    "3m_ec1000_cpl": (like_ec1000_cpl_3m, cosi_ec1000_cpl_3m, 3),
    "24m_ec200_cpl_pl": (like, cosi, 24),
    "24m_ec200_cpl": (like_cpl_only, cosi_cpl_only, 24),
    "24m_ec1000_cpl_pl": (like_ec1000, cosi_ec1000, 24),
    "24m_ec1000_cpl": (like_ec1000_cpl_only, cosi_ec1000_cpl_only, 24),
}
dual_ts = {label: cosi_source_detection_ts(fit, plugin)[0] for label, (fit, plugin, _) in fit_pairs.items()}
delta_3m_ec200 = exposure_model_improvement_ts(like_ec200_cpl_3m, like_ec200_3m)
delta_3m_ec1000 = exposure_model_improvement_ts(like_ec1000_cpl_3m, like_ec1000_3m)
delta_24m_ec200 = exposure_model_improvement_ts(like_cpl_only, like)
delta_24m_ec1000 = exposure_model_improvement_ts(like_ec1000_cpl_only, like_ec1000)

fit_summary_path = save_agn_fit_summary(
    output_path=FIT_SUMMARY_DIR / "NGC4151_Case2_CPL_PL_comparison_fit_summary_3_24Months.txt",
    fit_results={label: fit.results for label, (fit, _, _) in fit_pairs.items()},
    injected_models={
        "3m_ec200_cpl_pl": {"thermal": spectrum_inj_ec200, "nonthermal": spectrum_inj_ec200_PL},
        "3m_ec200_cpl": {"thermal": spectrum_inj_ec200},
        "3m_ec1000_cpl_pl": {"thermal": spectrum_inj_ec1000, "nonthermal": spectrum_inj_ec1000_PL},
        "3m_ec1000_cpl": {"thermal": spectrum_inj_ec1000},
        "24m_ec200_cpl_pl": {"thermal": spectrum_inj_ec200, "nonthermal": spectrum_inj_ec200_PL},
        "24m_ec200_cpl": {"thermal": spectrum_inj_ec200},
        "24m_ec1000_cpl_pl": {"thermal": spectrum_inj_ec1000, "nonthermal": spectrum_inj_ec1000_PL},
        "24m_ec1000_cpl": {"thermal": spectrum_inj_ec1000},
    },
    ts_values=dual_ts,
    exposure_months={label: exposure for label, (_, _, exposure) in fit_pairs.items()},
    extra_statistics={
        "3m_ec200_cpl_pl": {"Delta_TS_Cpl_plus_Pl_vs_Cpl": delta_3m_ec200},
        "3m_ec1000_cpl_pl": {"Delta_TS_Cpl_plus_Pl_vs_Cpl": delta_3m_ec1000},
        "24m_ec200_cpl_pl": {"Delta_TS_Cpl_plus_Pl_vs_Cpl": delta_24m_ec200},
        "24m_ec1000_cpl_pl": {"Delta_TS_Cpl_plus_Pl_vs_Cpl": delta_24m_ec1000},
    },
    notes="Case 2 three-month fits use the companions with the 24-month threshold norm_nt.",
)
print(f"Saved combined 3/24-month comparison summary: {fit_summary_path}")


## Global fit and representative simultaneous COSI SED

The COSI SED uses the saved representative realization and the fitted global spectral shape.

In [ ]:
results = like.results
results_ec1000 = like_ec1000.results
results_sed = results
results_sed_ec1000 = results_ec1000
data_sed_hist = cosi._data.copy()
bkg_sed_hist = cosi._raw_bkg_hist.copy()
data_sed_ec1000_hist = cosi_ec1000._data.copy()
bkg_sed_ec1000_hist = cosi_ec1000._raw_bkg_hist.copy()

print(results.display())

def make_composite_flux_propagator(fit_results, source_name):
    optimized_source = fit_results.optimized_model[source_name]
    thermal_shape, tail_shape = optimized_source.spectrum.main.shape.functions
    link_function = tail_shape.K.auxiliary_variable[1]

    # Evaluate the linked composite explicitly so the free link coefficient
    # and all cross-component covariances are included in each sample.
    def evaluate_at(energy, K_1, xc_1, b, index_2):
        thermal_flux = thermal_shape.evaluate_at(energy, K=K_1, xc=xc_1)
        tail_flux = (b * K_1) * (energy / tail_shape.piv.value) ** index_2
        return thermal_flux + tail_flux

    return fit_results.propagate(
        evaluate_at,
        K_1=fit_results.get_variates(thermal_shape.K.path),
        xc_1=fit_results.get_variates(thermal_shape.xc.path),
        b=fit_results.get_variates(link_function.b.path),
        index_2=fit_results.get_variates(tail_shape.index.path),
    )


results_err_source1 = make_composite_flux_propagator(results, "source1")
results_err_source2_ec1000 = make_composite_flux_propagator(results_ec1000, "source2")


In [ ]:
energy = np.geomspace(100 * u.keV, 10 * u.MeV).to_value(u.keV)

flux_lo = np.zeros_like(energy)
flux_median = np.zeros_like(energy)
flux_hi = np.zeros_like(energy)
flux_inj = np.zeros_like(energy)

flux_lo_ec1000 = np.zeros_like(energy)
flux_median_ec1000 = np.zeros_like(energy)
flux_hi_ec1000 = np.zeros_like(energy)
flux_inj_ec1000 = np.zeros_like(energy)

# Add arrays for individual components
flux_median_thermal_200 = np.zeros_like(energy)
flux_median_tail_200 = np.zeros_like(energy)

flux_median_thermal_1000 = np.zeros_like(energy)
flux_median_tail_1000 = np.zeros_like(energy)

shape_200 = results.optimized_model["source1"].spectrum.main.shape
thermal_shape_200, tail_shape_200 = shape_200.functions
shape_1000 = results_ec1000.optimized_model["source2"].spectrum.main.shape
thermal_shape_1000, tail_shape_1000 = shape_1000.functions

for i, e in enumerate(energy):
    # Propagate the total composite shape in one covariance-aware operation.
    flux_combined = results_err_source1(e)
    flux_combined_ec1000 = results_err_source2_ec1000(e)

    # Store best-fit component curves for optional plotting.
    flux_median_thermal_200[i] = thermal_shape_200.evaluate_at(e)
    flux_median_tail_200[i] = tail_shape_200.evaluate_at(e)
    flux_median_thermal_1000[i] = thermal_shape_1000.evaluate_at(e)
    flux_median_tail_1000[i] = tail_shape_1000.evaluate_at(e)
    
    # Get median and intervals from the combined distribution
    flux_median[i] = flux_combined.median
    flux_lo[i], flux_hi[i] = flux_combined.equal_tail_interval(cl=0.68)

    flux_median_ec1000[i] = flux_combined_ec1000.median
    flux_lo_ec1000[i], flux_hi_ec1000[i] = flux_combined_ec1000.equal_tail_interval(cl=0.68)
    
    flux_inj[i] = spectrum_inj_ec200_total.evaluate_at(e)
    flux_inj_ec1000[i] = spectrum_inj_ec1000_total.evaluate_at(e)

binned_energy_edges = NGC4151_ec200.axes['Em'].edges.value
binned_energy = np.array([])
bin_sizes = np.array([])
binned_energy_ec1000 = np.array([])

for i in range(len(binned_energy_edges)-1):
    binned_energy = np.append(binned_energy, (binned_energy_edges[i+1] + binned_energy_edges[i]) / 2)
    bin_sizes = np.append(bin_sizes, binned_energy_edges[i+1] - binned_energy_edges[i])
    
    binned_energy_ec1000 = np.append(binned_energy_ec1000, (binned_energy_edges[i+1] + binned_energy_edges[i]) / 2)


expectation = cosi._expected_counts['source1']
expectation_ec1000 = cosi_ec1000._expected_counts['source2']


## Simultaneous COSI BinnedSED

Each true-energy bin uses the complete spectral shape from its fitted global model. All response-bin normalizations are fitted together to the saved representative realization with one Poisson likelihood. The COSI background rate is fixed to the global fit; per-bin TS and limits profile the other bin normalizations.

In [ ]:
# The simultaneous BinnedSED fit is implemented in agn_binned_sed_fit_utils.py.
from agn_binned_sed_fit_utils import fit_global_binned_sed


In [ ]:
from agn_binned_sed_fit_utils import fit_global_binned_sed

thermal_cosi_sed_df = fit_global_binned_sed(
    global_results=results_sed, global_plugin=cosi,
    source_name='source1', dr_path=dr,
    sc_orientation=sc_orientation, longitude_deg=l, latitude_deg=b,
    background_pseudocount=BACKGROUND_PSEUDOCOUNT, label='case2_comp_ec203_24m',
)
display(thermal_cosi_sed_df[["bin_index", "e_min_keV", "e_max_keV", "ts_value", "plot_role"]])


In [ ]:
from agn_binned_sed_fit_utils import fit_global_binned_sed

thermal_cosi_sed_df_ec1000 = fit_global_binned_sed(
    global_results=results_sed_ec1000, global_plugin=cosi_ec1000,
    source_name='source2', dr_path=dr,
    sc_orientation=sc_orientation, longitude_deg=l, latitude_deg=b,
    background_pseudocount=BACKGROUND_PSEUDOCOUNT, label='case2_comp_ec1000_24m',
)
display(thermal_cosi_sed_df_ec1000[["bin_index", "e_min_keV", "e_max_keV", "ts_value", "plot_role"]])


### Plot A — representative median data set

SED bin values and point/upper-limit decisions come only from the selected representative realization.


In [ ]:
from agn_sed_ensemble import classify_representative_sed

# Plot A uses only the selected representative realization. Its own
# joint-fit profile TS decides detection/upper limit/omission; using the saved representative realization.
representative_thermal_cosi_sed_df_ec1000 = classify_representative_sed(thermal_cosi_sed_df_ec1000, detection_ts=4.0)
representative_thermal_cosi_sed_df = classify_representative_sed(thermal_cosi_sed_df, detection_ts=4.0)

print('Plot A: representative median data set; classification uses its own bin TS.')
FONT_SIZE = 25

plt.rcParams["agg.path.chunksize"] = 10000
plt.rcParams.update({"font.size": FONT_SIZE})
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams.update(
    {
        "font.weight": "550",
        "axes.titleweight": "550",
        "axes.labelweight": "550",
    }
)

def _make_xerr(df):
    return KEV_TO_MEV * np.vstack(
        [
            df["e_ref_keV"] - df["e_min_keV"],
            df["e_max_keV"] - df["e_ref_keV"],
        ]
    )

def _split_sed_detections(df, ts_threshold=4.0, min_plot_ts=1.0):
    if 'plot_role' in df.columns:
        roles = df['plot_role'].astype(str)
        return (
            df.loc[roles == 'detection'].copy(),
            df.loc[roles == 'upper_limit'].copy(),
        )

    plot_mask = np.ones(len(df), dtype=bool)
    upper_limit_mask = df['is_upper_limit'].copy()
    if 'ts_value' in df.columns:
        plot_mask = df['ts_value'] >= min_plot_ts
        upper_limit_mask = upper_limit_mask | (df['ts_value'] < ts_threshold)
    return (
        df.loc[plot_mask & ~upper_limit_mask].copy(),
        df.loc[plot_mask & upper_limit_mask].copy(),
    )

n_sed_bins_200 = len(representative_thermal_cosi_sed_df)
n_sed_bins_1000 = len(representative_thermal_cosi_sed_df_ec1000)

fig, ax = plt.subplots(figsize=(12, 9), constrained_layout=True)

ax.xaxis.set_tick_params(which='major', size=12, width=1.5, direction='in', top='on', pad=8, labelsize=FONT_SIZE)
ax.xaxis.set_tick_params(which='minor', size=6, width=1.5, direction='in', top='on', pad=8)
ax.yaxis.set_tick_params(which='major', size=12, width=1.5, direction='in', right='on', pad=8, labelsize=FONT_SIZE)
ax.yaxis.set_tick_params(which='minor', size=6, width=1.5, direction='in', right='on', pad=8)

ax.spines['right'].set_visible(True)
ax.spines['top'].set_visible(True)

det_200, ul_200 = _split_sed_detections(representative_thermal_cosi_sed_df)

det_1000, ul_1000 = _split_sed_detections(representative_thermal_cosi_sed_df_ec1000)

ax.plot(
    energy * KEV_TO_MEV,
    SED_KEV_TO_ERG * energy * energy * flux_inj,
    color='#D55E00',
    ls=':',
    lw=3,
    label='Injected thermal model ($E_{c}$ = 200 keV)',
)
ax.plot(
    energy * KEV_TO_MEV,
    SED_KEV_TO_ERG * energy * energy * flux_median,
    color='#D55E00',
    lw=2,
    label='Global COSI-only fit ($E_{c}$ = 200 keV)',
)
ax.fill_between(
    energy * KEV_TO_MEV,
    SED_KEV_TO_ERG * energy * energy * flux_lo,
    SED_KEV_TO_ERG * energy * energy * flux_hi,
    alpha=0.18,
    color='#D55E00',
)

ax.plot(
    energy * KEV_TO_MEV,
    SED_KEV_TO_ERG * energy * energy * flux_inj_ec1000,
    color='#2A8BC3',
    ls=':',
    lw=3,
    label='Injected thermal model ($E_{c}$ = 1000 keV)',
)
ax.plot(
    energy * KEV_TO_MEV,
    SED_KEV_TO_ERG * energy * energy * flux_median_ec1000,
    color='#2A8BC3',
    lw=2,
    label='Global COSI-only fit ($E_{c}$ = 1000 keV)',
)
ax.fill_between(
    energy * KEV_TO_MEV,
    SED_KEV_TO_ERG * energy * energy * flux_lo_ec1000,
    SED_KEV_TO_ERG * energy * energy * flux_hi_ec1000,
    alpha=0.18,
    color='#2A8BC3',
)

if not det_200.empty:
    detection_xerr_200 = _make_xerr(det_200)
    detection_yerr_200 = np.vstack(
        [
            np.maximum(det_200['sed_erg_cm2_s'] - det_200['sed_lo_erg_cm2_s'], 0),
            np.maximum(det_200['sed_hi_erg_cm2_s'] - det_200['sed_erg_cm2_s'], 0),
        ]
    )

    ax.errorbar(
        det_200['e_ref_keV'] * KEV_TO_MEV,
        det_200['sed_erg_cm2_s'],
        xerr=detection_xerr_200,
        yerr=detection_yerr_200,
        fmt='o',
        color='#D55E00',
        ecolor='#D55E00',
        markerfacecolor='white',
        markeredgecolor='#D55E00',
        markeredgewidth=1.6,
        elinewidth=1.5,
        capsize=4,
        markersize=8,
        label=f'COSI SED ($E_{{c}}$ = 200 keV; {len(det_200)} detections out of {n_sed_bins_200} bins)',
    )

if not ul_200.empty:
    upper_xerr_200 = _make_xerr(ul_200)
    upper_y_200 = ul_200['sed_hi_erg_cm2_s'].to_numpy()
    upper_yerr_200 = np.maximum(0.5 * upper_y_200, np.finfo(float).tiny)

    ax.errorbar(
        ul_200['e_ref_keV'] * KEV_TO_MEV,
        upper_y_200,
        xerr=upper_xerr_200,
        yerr=upper_yerr_200,
        uplims=True,
        fmt='v',
        color='#D55E00',
        ecolor='#D55E00',
        markerfacecolor='white',
        markeredgecolor='#D55E00',
        markeredgewidth=1.8,
        elinewidth=2.4,
        capsize=6,
        markersize=11,
        barsabove=True,
        zorder=6,
        label='_nolegend_',
    )

if not det_1000.empty:
    detection_xerr_1000 = _make_xerr(det_1000)
    detection_yerr_1000 = np.vstack(
        [
            np.maximum(det_1000['sed_erg_cm2_s'] - det_1000['sed_lo_erg_cm2_s'], 0),
            np.maximum(det_1000['sed_hi_erg_cm2_s'] - det_1000['sed_erg_cm2_s'], 0),
        ]
    )

    ax.errorbar(
        det_1000['e_ref_keV'] * KEV_TO_MEV,
        det_1000['sed_erg_cm2_s'],
        xerr=detection_xerr_1000,
        yerr=detection_yerr_1000,
        fmt='o',
        color='#2A8BC3',
        ecolor='#2A8BC3',
        markerfacecolor='white',
        markeredgecolor='#2A8BC3',
        markeredgewidth=1.6,
        elinewidth=1.5,
        capsize=4,
        markersize=8,
        label=f'COSI SED ($E_{{c}}$ = 1000 keV; {len(det_1000)} detections out of {n_sed_bins_1000} bins)',
    )

if not ul_1000.empty:
    upper_xerr_1000 = _make_xerr(ul_1000)
    upper_y_1000 = ul_1000['sed_hi_erg_cm2_s'].to_numpy()
    upper_yerr_1000 = np.maximum(0.5 * upper_y_1000, np.finfo(float).tiny)

    ax.errorbar(
        ul_1000['e_ref_keV'] * KEV_TO_MEV,
        upper_y_1000,
        xerr=upper_xerr_1000,
        yerr=upper_yerr_1000,
        uplims=True,
        fmt='v',
        color='#2A8BC3',
        ecolor='#2A8BC3',
        markerfacecolor='white',
        markeredgecolor='#2A8BC3',
        markeredgewidth=1.8,
        elinewidth=2.4,
        capsize=6,
        markersize=11,
        barsabove=True,
        zorder=6,
        label='_nolegend_',
    )

legend_handles = [
    plt.Line2D([], [], color='#D55E00', ls=':', lw=3, label='Injected ($E_{c, th}$ = 200 keV)'),
    plt.Line2D([], [], color='#D55E00', lw=2, label='Best fit & 68% band'),
    plt.Line2D([], [], color='#2A8BC3', ls=':', lw=3, label='Injected ($E_{c, th}$ = 1000 keV)'),
    plt.Line2D([], [], color='#2A8BC3', lw=2, label='Best fit & 68% band'),
]

if not det_200.empty or not det_1000.empty:
    legend_handles.append(
        plt.Line2D([], [], color='0.25', marker='o', markerfacecolor='white', markeredgewidth=1.6, linestyle='None', markersize=8, label='COSI SED')
    )

ax.text(
    0.98, 0.97,
    f'NGC 4151\n Th. + Lep. (Case 2)\n{exposure_4151}-month',
    transform=ax.transAxes,
    ha='right', va='top',
    fontsize=FONT_SIZE,
    fontweight='550'
)

ax.set_xscale('log')
ax.set_yscale('log')

ax.set_ylim(1e-4 * SED_KEV_TO_ERG, 1 * SED_KEV_TO_ERG)

ax.set_xlim((85) * KEV_TO_MEV, (12000) * KEV_TO_MEV)

# ax.xaxis.set_major_locator(mticker.FixedLocator([0.1, 5, 10]))
# ax.xaxis.set_major_formatter(mticker.FixedFormatter(["0.1", "5", "10"]))

ax.set_xlabel('Energy (MeV)', fontsize=FONT_SIZE)
ax.set_ylabel(r'Energy Flux (erg cm$^{-2}$ s$^{-1}$)', fontsize=FONT_SIZE)
ax.legend(handles=legend_handles, fontsize=FONT_SIZE, loc='lower left', frameon=False)

save_path = f'/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/Papers/AGN_Corona_EC_New/Plots/Fluctuate_True/NGC4151_Cpl_Pl_Comparison_Fit_200_1000_{exposure_4151}Months_SED_RepresentativeMedianDataset.pdf'
Path(save_path).parent.mkdir(parents=True, exist_ok=True)
fig.savefig(save_path, bbox_inches='tight')
plt.show()
